# 02 - Bronze Layer: Claims Data (Autoloader)

**Project:** Auto Insurance Claims & Telematics Analytics

## What this notebook does
Ingests the auto insurance claims CSV into a bronze Delta table using
Autoloader, same trigger(availableNow=True) batch-style pattern used in the
Healthcare and P&C projects for file-based sources.

## Table created
- `main.auto_insurance_telematics.bronze_claims`

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

claims_schema = StructType([
    StructField("months_as_customer", IntegerType(), True),
    StructField("age", IntegerType(), True),
    StructField("policy_number", IntegerType(), True),
    StructField("policy_bind_date", DateType(), True),
    StructField("policy_state", StringType(), True),
    StructField("policy_csl", StringType(), True),
    StructField("policy_deductable", IntegerType(), True),
    StructField("policy_annual_premium", DoubleType(), True),
    StructField("umbrella_limit", IntegerType(), True),
    StructField("insured_zip", IntegerType(), True),
    StructField("insured_sex", StringType(), True),
    StructField("insured_education_level", StringType(), True),
    StructField("insured_occupation", StringType(), True),
    StructField("insured_hobbies", StringType(), True),
    StructField("insured_relationship", StringType(), True),
    StructField("capital-gains", IntegerType(), True),
    StructField("capital-loss", IntegerType(), True),
    StructField("incident_date", DateType(), True),
    StructField("incident_type", StringType(), True),
    StructField("collision_type", StringType(), True),
    StructField("incident_severity", StringType(), True),
    StructField("authorities_contacted", StringType(), True),
    StructField("incident_state", StringType(), True),
    StructField("incident_city", StringType(), True),
    StructField("incident_location", StringType(), True),
    StructField("incident_hour_of_the_day", IntegerType(), True),
    StructField("number_of_vehicles_involved", IntegerType(), True),
    StructField("property_damage", StringType(), True),
    StructField("bodily_injuries", IntegerType(), True),
    StructField("witnesses", IntegerType(), True),
    StructField("police_report_available", StringType(), True),
    StructField("total_claim_amount", IntegerType(), True),
    StructField("injury_claim", IntegerType(), True),
    StructField("property_claim", IntegerType(), True),
    StructField("vehicle_claim", IntegerType(), True),
    StructField("auto_make", StringType(), True),
    StructField("auto_model", StringType(), True),
    StructField("auto_year", IntegerType(), True),
    StructField("fraud_reported", StringType(), True),
    StructField("_c39", StringType(), True),
])

In [0]:
from pyspark.sql.functions import current_timestamp, col

def autoload_csv_to_bronze(source_path: str, table_name: str, schema: StructType):
    """
    Incrementally ingest CSV files from a Volume path into a bronze Delta table
    using Autoloader (cloudFiles). trigger(availableNow=True) processes all
    files currently sitting in the source path and then stops - ideal for a
    scheduled job rather than an always-on cluster.
    """
    checkpoint_path = f"/Volumes/main/auto_insurance_telematics/checkpoints/{table_name}"
    schema_path = f"/Volumes/main/auto_insurance_telematics/checkpoints/{table_name}_schema"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", "true")
        .schema(schema)
        .load(source_path)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"main.auto_insurance_telematics.{table_name}")
    )

In [0]:
autoload_csv_to_bronze(
    source_path="/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims*.csv",
    table_name="bronze_claims",
    schema=claims_schema,
)

In [0]:
%sql
SELECT COUNT(*) FROM main.auto_insurance_telematics.bronze_claims;

COUNT(*)
1000


In [0]:
%sql
SELECT * FROM main.auto_insurance_telematics.bronze_claims LIMIT 5;

months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39,_ingested_at,_source_file
328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,MALE,MD,craft-repair,sleeping,husband,53300,0,2015-01-25,Single Vehicle Collision,Side Collision,Major Damage,Police,SC,Columbus,9935 4th Drive,5,1,YES,1,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y,null,2026-08-29T21:52:55.768Z,/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims.csv
228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,MALE,MD,machine-op-inspct,reading,other-relative,0,0,2015-01-21,Vehicle Theft,?,Minor Damage,Police,VA,Riverwood,6608 MLK Hwy,8,1,?,0,0,?,5070,780,780,3510,Mercedes,E400,2007,Y,null,2026-08-29T21:52:55.768Z,/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims.csv
134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,FEMALE,PhD,sales,board-games,own-child,35100,0,2015-02-22,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Columbus,7121 Francis Lane,7,3,NO,2,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,null,2026-08-29T21:52:55.768Z,/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims.csv
256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,FEMALE,PhD,armed-forces,board-games,unmarried,48900,-62400,2015-01-10,Single Vehicle Collision,Front Collision,Major Damage,Police,OH,Arlington,6956 Maple Drive,5,1,?,1,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,null,2026-08-29T21:52:55.768Z,/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims.csv
228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,MALE,Associate,sales,board-games,unmarried,66000,-46000,2015-02-17,Vehicle Theft,?,Minor Damage,None,NY,Arlington,3041 3rd Ave,20,1,NO,0,1,NO,6500,1300,650,4550,Accura,RSX,2009,N,null,2026-08-29T21:52:55.768Z,/Volumes/main/auto_insurance_telematics/raw_data/insurance_claims.csv
